# Christenson: Parse (sentence-level)

OHCO: `chap_num, sent_num, token_num`

Source: `christenson-LINE_LITERAL-with-chaps.csv` — joins lines to chapters, splits on punctuation.

In [ ]:
import pandas as pd
import re

In [ ]:
src_id = 'christenson'
lines_path = '../../notebooks/christenson/christenson-LINE_LITERAL-with-chaps.csv'

## Load lines and group into chapters

In [ ]:
LINES = pd.read_csv(lines_path)
LINES = LINES.dropna(subset=['quc_str', 'chap_num'])
CHAP = (LINES.groupby('chap_num')['quc_str']
        .apply(lambda x: ' '.join(x.astype(str)))
        .reset_index().rename(columns={'quc_str': 'doc_str'}))
print(f'{len(CHAP)} chapters')
CHAP.head()

## CHAP to SENT — split on sentence-terminal punctuation

In [ ]:
SENT = (
    CHAP
    .assign(doc_str=lambda df: df.doc_str.str.split(r'(?<=[.!?])\s+'))
    .explode('doc_str').dropna(subset=['doc_str'])
)
SENT = SENT[SENT.doc_str.str.strip() != ''].copy()
SENT['sent_num'] = SENT.groupby('chap_num').cumcount()
SENT = SENT.reset_index(drop=True); SENT.index.name = 'doc_id'
DOC = SENT[['doc_str']]; DOCMAP = SENT[['chap_num', 'sent_num']]
print(f'{len(DOC):,} sentences from {DOCMAP.chap_num.nunique()} chapters')
DOCMAP.head()

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", '', regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f'{src_id}-TOKEN.csv')
DOC.to_csv(f'{src_id}-DOC.csv')
DOCMAP.to_csv(f'{src_id}-DOCMAP.csv')
print('Saved to notebooks/doc_tables/')